# AI Cyber Threat Detection - Alert Summary

This notebook converts machine learning predictions into simple cybersecurity alert summaries.

The goal is to demonstrate how AI-based threat detection can support SOC analysts by turning model outputs into clear, human-readable alerts.

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [2]:
data = fetch_kddcup99(percent10=True, as_frame=True)
df = data.frame

df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,labels
0,0,b'tcp',b'http',b'SF',181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,b'normal.'
1,0,b'tcp',b'http',b'SF',239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,b'normal.'
2,0,b'tcp',b'http',b'SF',235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,b'normal.'
3,0,b'tcp',b'http',b'SF',219,1337,0,0,0,0,...,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,b'normal.'
4,0,b'tcp',b'http',b'SF',217,2032,0,0,0,0,...,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,b'normal.'


In [3]:
# 0 = normal network activity
# 1 = suspicious / attack activity

df["target"] = df["labels"].apply(lambda x: 0 if x == b"normal." else 1)

df["target"].value_counts()

,count
target,
1,396743
0,97278


In [4]:
X = df.drop(columns=["labels", "target"])
y = df["target"]

In [5]:
X_sample, _, y_sample, _ = train_test_split(
    X,
    y,
    train_size=100000,
    random_state=42,
    stratify=y
)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    random_state=42,
    stratify=y_sample
)

In [7]:
categorical_features = X_train.select_dtypes(include=["object"]).columns
numeric_features = X_train.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [8]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_she...
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate'],
      dtype='object')),
                                                 ('numeric', 'passthrough',
                                                  Index([], dtype='object'))])),
                ('model',
                 RandomForestClassifier(n_estimators=50, n_jobs=-1,
                                        random_state=42))])

In [9]:
predictions = pipeline.predict(X_test)
probabilities = pipeline.predict_proba(X_test)

confidence_scores = probabilities.max(axis=1)

In [10]:
alerts = X_test.copy()

alerts["predicted_class"] = predictions
alerts["confidence"] = confidence_scores
alerts["true_class"] = y_test.values

alerts.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,predicted_class,confidence,true_class
186343,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,1.0,1
325074,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,1.0,1
256062,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,1.0,1
461833,0,b'tcp',b'private',b'REJ',0,0,0,0,0,0,...,0.07,0.0,0.0,0.0,0.0,1.0,1.0,1,1.0,1
17618,0,b'tcp',b'http',b'SF',244,28147,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0


In [11]:
def label_prediction(value):
    if value == 0:
        return "Normal Activity"
    else:
        return "Suspicious Activity"


alerts["prediction_label"] = alerts["predicted_class"].apply(label_prediction)
alerts["true_label"] = alerts["true_class"].apply(label_prediction)

alerts[["prediction_label", "confidence", "true_label"]].head()

,prediction_label,confidence,true_label
186343,Suspicious Activity,1.0,Suspicious Activity
325074,Suspicious Activity,1.0,Suspicious Activity
256062,Suspicious Activity,1.0,Suspicious Activity
461833,Suspicious Activity,1.0,Suspicious Activity
17618,Normal Activity,1.0,Normal Activity


In [12]:
def generate_alert_summary(row):
    prediction = row["prediction_label"]
    confidence = row["confidence"]

    if prediction == "Suspicious Activity":
        if confidence >= 0.90:
            severity = "High"
            action = "Immediate investigation recommended."
        elif confidence >= 0.70:
            severity = "Medium"
            action = "Review the event and compare it with recent network activity."
        else:
            severity = "Low"
            action = "Monitor the activity and investigate if similar events continue."

        summary = (
            f"Alert: Suspicious network activity detected.\n"
            f"Severity: {severity}\n"
            f"Model confidence: {confidence:.2f}\n"
            f"Recommended action: {action}"
        )

    else:
        summary = (
            f"Alert: No suspicious activity detected.\n"
            f"Severity: Informational\n"
            f"Model confidence: {confidence:.2f}\n"
            f"Recommended action: No immediate action required."
        )

    return summary

In [13]:
alerts["alert_summary"] = alerts.apply(generate_alert_summary, axis=1)

alerts[["prediction_label", "confidence", "alert_summary"]].head()

,prediction_label,confidence,alert_summary
186343,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
325074,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
256062,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
461833,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
17618,Normal Activity,1.0,Alert: No suspicious activity detected.\nSever...


In [14]:
suspicious_alerts = alerts[alerts["predicted_class"] == 1]

suspicious_alerts[["prediction_label", "confidence", "alert_summary"]].head(10)

,prediction_label,confidence,alert_summary
186343,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
325074,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
256062,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
461833,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
159435,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
198159,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
236749,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
102457,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
222101,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...
277146,Suspicious Activity,1.0,Alert: Suspicious network activity detected.\n...


In [15]:
for i, alert in enumerate(suspicious_alerts["alert_summary"].head(5), start=1):
    print(f"--- Alert {i} ---")
    print(alert)
    print()

--- Alert 1 ---
Alert: Suspicious network activity detected.
Severity: High
Model confidence: 1.00
Recommended action: Immediate investigation recommended.

--- Alert 2 ---
Alert: Suspicious network activity detected.
Severity: High
Model confidence: 1.00
Recommended action: Immediate investigation recommended.

--- Alert 3 ---
Alert: Suspicious network activity detected.
Severity: High
Model confidence: 1.00
Recommended action: Immediate investigation recommended.

--- Alert 4 ---
Alert: Suspicious network activity detected.
Severity: High
Model confidence: 1.00
Recommended action: Immediate investigation recommended.

--- Alert 5 ---
Alert: Suspicious network activity detected.
Severity: High
Model confidence: 1.00
Recommended action: Immediate investigation recommended.



## Result

This notebook demonstrates how machine learning predictions can be converted into simple SOC-style alert summaries.

Instead of only showing raw model outputs, the project presents the prediction in a format that a security analyst could understand more easily.

This is a basic first version of SOC automation and can later be improved with:

- More detailed alert explanations
- Feature importance
- Attack type classification
- Risk scoring
- LLM-based alert summarization
- Dashboard integration